<center><b style="font-size:1.17em;">Simulations</b></center>
<div style="font-size:6px; line-height:8px;">&nbsp;</div>
<center>
Suprathreshold clustered <strong>glutamatergic</strong> input and temporally varying fast <strong>GABA<sub>A</sub></strong> synaptic receptor inputs
</center>

- **2810** – ctrl
- **2811** – same as 2810 but increased temporal fidelity; do not record all 3D
- **2812** – same as ctrl but M1R activation
- **2813** – same as 2812 but increased temporal fidelity; do not record all 3D
- **2818** – same as ctrl but M1R activation and reduced to 13 clustered glutamatergic inputs
- **2819** – same as 2818 but increased temporal fidelity; do not record all 3D
- **28110** – ctrl and block Naf
- **28111** – same as 2810 but increased temporal fidelity; do not record all 3D and block Naf
- **28112** – ctrl M1R activation and block Naf
- **28113** – same as 28112 but increased temporal fidelity; do not record all 3D and block Naf
- **28118** – same as ctrl but M1R activation and iSPN reduced to 13 clustered glutamatergic inputs and block Naf
- **28119** – same as 28118 but increased temporal fidelity; do not record all 3D and block Naf

**for dspns only blocking Kv7**

In **dSPN** and **iSPN**, 15 and 14 **clustered glutamatergic inputs** are the minimum required to elicit an upstate at each site

In **dSPN** and **iSPN** and reduced K+ conditions (**KCNQ only** and **KCNQ + Kir**, respectively), an upstate is observed with **15 and 14 clustered glutamatergic inputs** for either cell types, respectively.

In iSPN and reduced K+ conditions (**KCNQ + Kir + Kaf**), an upstate is observed with 13 clustered glutamatergic inputs.


**Channel-specific blocking rules**

- If **dSPN** → block **KCNQ only**
- If **iSPN** → block **KCNQ + Kir** OR **KCNQ + Kir + Kv4**

---


In [ ]:
# choose sim and cell_type (can be 'dspn' or 'ispn)
sim = '2811'
cell_type = 'ispn'

In [ ]:
############################################## SETTINGS ##############################################  
# settings
model = 3

# import default settings
import os, sys

cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

os.chdir(main_dir)
sys.path.insert(0, main_dir)

%run -i settings.py

# # alter default settings if necessary
if sim in ['2810', '2812', '2818', '28110', '28118']:
    record_3D = True
    record_3D = True
    record_3D_mechs = True
    record_3D_Ca = True
    record_Ca = False
    record_currents = True
    record_branch = True 
    record_mechs = True
    record_path_dend = True 
    record_path_spines = False   
    # record_3D_impedance = True  
    
record_branch = False


dend_glut = ['dend[15]']

glut = True

if compare_last_digit(sim, 2) or compare_last_digit(sim, 3) or compare_last_digit(sim, 8) or compare_last_digit(sim, 9) or compare_last_digit(sim, 12) or compare_last_digit(sim, 13):
    if cell_type == 'dspn':
        scale_factors = {'kcnq': 0.5}
    else:
        scale_factors = {'kcnq': 0.5, 'kir': 0.5, 'kaf': 0.5}
    v_init = -85.4087

if compare_last_digit(sim, 12) or compare_last_digit(sim, 13) or compare_last_digit(sim, 18) or compare_last_digit(sim, 19):
    if cell_type == 'dspn':
        scale_factors = {'naf': 0, 'kcnq': 0.5}
    else:
        scale_factors = {'naf': 0, 'kcnq': 0.5, 'kir': 0.5, 'kaf': 0.5}
    v_init = -85.4087

if compare_last_digit(sim, 10) or compare_last_digit(sim, 11):
    scale_factors = {'naf': 0}


    
gaba = True
glut_frequency = 1000 # every 1ms
AMPA = True
NMDA = True
g_AMPA = 0.001

# generate rel_glut_onsets
dstep1 = int(1/glut_frequency *1e3)

if dstep1 > 0:
    rel_glut_onsets = list(range(0, num_gluts * dstep1, dstep1))
else:
    rel_glut_onsets = 0 * num_gluts  # num_gluts repetitions of glut_time

# Phasic gaba
gaba = True
gaba_frequency = 50 
g_GABA = 0.001
gaba_reversal = -60
vary_gaba_time = False # if true timing of gaba is varied relative to glut; if false then vary glut relative to gaba
stim_time = 200
    
# recommended:
timing_range = range(100,301,10)
if sim in ['2811', '2813', '2819', '28111', '28113', '28119']:
    timing_range = list(range(100,301,1))

timing_range = np.insert(timing_range, 0, [stim_time, stim_time])
gaba_range = np.repeat(True, len(timing_range)-1) 
gaba_range = np.insert(gaba_range, 0, False)
glut_range = np.repeat(True, len(timing_range)-1) 
glut_range = np.insert(glut_range, 1, False)
Nsims = len(timing_range)

sim_time = 600
save = True
dt =  0.025
deltat = dt * ds

In [ ]:
# Morphology
from analysis_functions import *

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove)
cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])


cell_coordinates = np.array(cell_coordinates, dtype=object)

width=600
height=600
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='grey', height=height, width=width)

fig_morphology.show()

In [ ]:
# chose multiple input sites for upstate generation ~ 157.5 um from the soma
cell_coordinates = np.array(cell_coordinates, dtype=object)

# Target distance
dist_target = 157.5

# Prepare an empty list to store results
closest_points = []

# Iterate over unique section names
for sec_name in np.unique(cell_coordinates[:, 0]):
    # Filter rows for the current `sec.name()`
    sec_rows = cell_coordinates[cell_coordinates[:, 0] == sec_name]
    
    # Extract distances for the current section
    distances = sec_rows[:, 5].astype(float)
    
    # Check if the target distance is within the range of this section's distances
    if distances.min() <= dist_target <= distances.max():
        # Calculate absolute differences from the target
        abs_diff = np.abs(distances - dist_target)
        
        # Find the index of the minimum difference
        min_index = np.argmin(abs_diff)
        
        # Append the closest row to the results
        closest_points.append(sec_rows[min_index])

# Convert the results to a numpy array for convenience
# Nsites = 5 # sites to generate upstates

synapses_ = np.array(closest_points, dtype=object)
selected_dendrites_df = pd.DataFrame(synapses_)

# # if random then do this:
# selected_dendrites_df = selected_dendrites_df.sample(frac=1, random_state=316309).reset_index(drop=True)
# dend_glut_full = selected_dendrites_df.iloc[0:Nsites][0].tolist()
# glut_locations_full = selected_dendrites_df.iloc[0:Nsites][1].tolist()

if cell_type == 'dspn':
    dend_glut_full = ['dend[15]', 'dend[52]', 'dend[47]', 'dend[21]'] # dend_glut_full = ['dend[15]', 'dend[52]', 'dend[28]', 'dend[47]', 'dend[57]']
    num_gluts = 15
elif cell_type == 'ispn':
    dend_glut_full = ['dend[18]', 'dend[33]', 'dend[5]', 'dend[11]']
    num_gluts = 14

if compare_last_digit(sim, 8) or compare_last_digit(sim, 9) or compare_last_digit(sim, 18) or compare_last_digit(sim, 19):
    num_gluts = 13

    
Nsites = len(dend_glut_full)

glut_locations_full = selected_dendrites_df.set_index(0).loc[dend_glut_full, 1].tolist()

dend_glut = [dend for dend in dend_glut_full for _ in range(num_gluts)]

glutamate_locations=[]
for dend, loc in zip(dend_glut_full, glut_locations_full):
    result = spine_locator(cell_type=cell_type, specs=specs, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, 
                           dend_glut=[dend], num_gluts=num_gluts, method=method, dend2remove=dend2remove, rel_x=loc)
    glutamate_locations.extend(result)

# generate rel_glut_onsets
dstep1 = int(1/glut_frequency *1e3)

if dstep1 > 0:
    rel_glut_onsets = list(range(0, num_gluts * dstep1, dstep1))
else:
    rel_glut_onsets = 0 * num_gluts  # num_gluts repetitions of glut_time

# Nsites is number of separate upstate locations
if Nsites > 1:
    rel_glut_onsets = rel_glut_onsets * Nsites
    num_gluts = num_gluts * Nsites

In [ ]:
# select synapses at random to sample from
from analysis_functions import *

# dend2remove1 = ['dend[55]', 'dend[19]', 'dend[42]']
dend2remove1 = None

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove1)
cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])


cell_coordinates = np.array(cell_coordinates, dtype=object)

width = 600
height = 600
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='grey', height=height, width=width)

fig_morphology.show()

# choose inputs
cell_coordinates_df = pd.DataFrame(cell_coordinates, columns=['section', 'rel.x', 'x', 'y', 'z', 'dist', 'diam'])
dend_df = cell_coordinates_df[cell_coordinates_df.iloc[:, 0] != 'soma[0]']

# sample N=154 rows without replacement
sampled_df = dend_df.sample(n=154, replace=False, random_state=1)
sampled_df = sampled_df.sort_index()
indices = sampled_df.index.to_list()
N_GABA_total = len(indices)

dend_gaba_full = cell_coordinates_df.loc[indices, 'section'].tolist()
gaba_locations_full = cell_coordinates_df.loc[indices, 'rel.x'].tolist()

pairs = list(zip(dend_gaba_full, gaba_locations_full))
random.seed(1)  
random.shuffle(pairs)
dend_gaba_full, gaba_locations_full = zip(*pairs)

# convert back to lists
dend_gaba_full = list(dend_gaba_full)
gaba_locations_full = list(gaba_locations_full)

In [ ]:
sims281x = ['2810', '2811', '2812', '2813', '2818', '2819', '28110', '28111', '28112', '28113', '28118', '28119']

soma_v_master = []
dend_v_master = []

# Determine which list 'sim' belongs to
sim_list=None
sim_list = [s for s in sims281x if s == sim]
    
if sim_list:
    for sim in sim_list:
    
        num_gabas = 40
        dend_gaba = dend_gaba_full[0:num_gabas]
        gaba_locations = gaba_locations_full[0:num_gabas]


        record_dendrite = dend_glut[0]
        record_location = [glutamate_locations[0]]
    
        rel_gaba_onsets = rel_gaba_onset(n=num_gabas, N=num_gabas)
        for ii in tqdm(list(range(Nsims))): # for ii in tqdm(list(range(183,Nsims))): # for ii in tqdm(list(range(Nsims-1, Nsims))): # for ii in tqdm(list(range(Nsims))): # for ii in tqdm(list(range(182,198))):
            # to store sim-generated variables
            data_dict = {
                'v_all_3D': {},
                'Ca_all_3D': {},
                'imp_all_3D': {},
                'i_mechs_3D': {},
                'v_dend_tree': {},
                'v_spine_tree': {},
                'v_branch': {},
                'vsoma': [],
                'vdend': [],
                'v_dend_activated': {},
                'vspine': [],
                'v_spine_activated': {},
                'Ca_soma': [],
                'Ca_dend': [],
                'Ca_spine': [],
                'timing': [],
                'dists': [],
                'dists_spine': [],
                'i_mechs_dend': {},
                'i_mechs_dend_tree': {},
                'i_mechs_spine_tree': {},
                'i_ampa': {},
                'i_nmda': {},
                'i_gaba': {},
                'g_gaba': {},
                'record_dists':[],
                'record_spine':[],
                'spine_dist': [],
                'capacitance': [],
                'timestamp': {}
                }
    
            # get coordinates for all unique segments within all sections
            cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove)
    
            # scale conductances
            if scale_factors is not None:
                printed = set()  # track which mechanisms have already been reported
                for sec in h.allsec():
                    for seg in sec:
                        for mech, sf in scale_factors.items():
                            if hasattr(seg, mech):
                                getattr(seg, mech).sf = sf
                                if mech not in printed:
                                    if mech in ['cal12', 'cal13', 'can', 'car', 'cav32', 'cav33']:
                                        print(f"permeability (cm/s) scaled: '{mech}' scaling factor = {sf}")
                                    else:
                                        print(f"conductance (S/cm²) scaled: '{mech}' scaling factor = {sf}")
                                    printed.add(mech)

     
            mechs=['pas', 'kdr', 'naf', 'kaf', 'kas', 'kir', 'kcnq', 'cal12', 'cal13', 'can', 'car', 'cav32', 'cav33', 'sk', 'bk']
            mech_distr_3D = record_mechs_distr_3D(cell=cell, mechs=mechs)
    
            # set these to False is they are not already assigned
            glut_reversal = globals().get('glut_reversal', 0) 
            vary_location = globals().get('vary_location', False)  
            dend_glut_range = globals().get('dend_glut_range', None)  
            vary_gaba_location = globals().get('vary_gaba_location', False)   
            
            # rec_all_path = globals().get('rec_all_path', False)  # default is False
            # if False records all v from spine / dendrite to soma
            # if True records at all unique sites including those distal to the synapse
    
            # determine if axoshaft or axospinous glutamatergic synapse
            axoshaft = globals().get('axoshaft', False)
            axospine = not axoshaft    
    
            # Common parameters
            common_params = {
                'cell_type': cell_type, 
                'specs': specs, 
                'spine_per_length':spine_per_length,
                'frequency': frequency,
                'd_lambda': d_lambda,
                'glut_reversal': glut_reversal,
                'num_gluts': num_gluts,
                'gaba_reversal': gaba_reversal,
                'num_gabas': num_gabas,
                'sim_time': stim_time,
                'dt': dt,
                'v_init': v_init,
                'dend2remove': dend2remove,
                'neck_dynamics': neck_dynamics,
                'soma_diameter': soma_diameter
            }
    
            if not vary_location:
                syn_reversals_params = {
                    'dend_glut': dend_glut,
                    'glutamate_locations': glutamate_locations,
                    'dend_gaba': dend_gaba,
                    'gaba_locations': gaba_locations,
                    **common_params
                }
                glut_reversals, gaba_reversals = syn_reversals(**syn_reversals_params)
    
            else:
                syn_reversals_params = {
                    **common_params
                }
                if vary_gaba_location:
                    syn_reversals_params.update({
                        'dend_glut': dend_glut,
                        'glutamate_locations': glut_locations,
                        'dend_gaba': dend_gaba_range,
                        'gaba_locations': gaba_locations_range,
                    })
                    glut_reversals, gaba_reversals_range = syn_reversals(**syn_reversals_params)
                    gaba_reversals_range = gaba_reversals_range * len(gaba_locations_range)
                else:
                    syn_reversals_params.update({
                        'dend_glut': dend_glut_range,
                        'glutamate_locations': glut_locations_range,
                        'dend_gaba': dend_gaba,
                        'gaba_locations': gaba_locations,
                    })
                    glut_reversals_range, gaba_reversals = syn_reversals(**syn_reversals_params)
                    glut_reversals_range = glut_reversals_range * len(glut_locations_range)
    
    
            # perform Nsim simulations 
        
            # time stamp date
            time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

            sim_image_path = None
            if Nsim_save:
                sim_image_path = 'downsample/{}/model{}/{}/images/sim{}/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim, ii)
                os.makedirs(sim_image_path, exist_ok=True)

            # set variables for this simulation
            timing = timing_range[ii]
            gaba = gaba_range[ii]
            glut = glut_range[ii]
            if vary_gaba_time:
                glut_time = stim_time
                gaba_time = timing
            else:
                glut_time = timing
                gaba_time = stim_time
            if vary_location:
                if vary_gaba_location: 
                    dend_gaba = [dend_gaba_range[ii]]
                    gaba_reversals = [gaba_reversals_range[ii]] if num_gabas == 1 else gaba_reversals_range[ii]

                else:
                    dend_glut = [dend_glut_range[ii]]
                    glut_reversals = [glut_reversals_range[ii]] if num_gluts == 1 else glut_reversals_range[ii]

                record_dendrite = record_dends[ii]
                glutamate_locations = [glut_locations_range[ii]]
                gaba_locations = [gaba_locations_range[ii]]
                record_location = [record_locs[ii]]

            if tonic:
                gbar_gaba = gbar_gaba_range[ii]
                print('tonic GABA ', gbar_gaba)
            else:
                gbar_gaba = 0

            # label from 1 to N+1 for R analysis
            protocol = 'sim{}'.format(ii+1)

            metadata = {}
            # Add fixed variables to metadata
            for name in variable_names:
                try:
                    if name not in metadata:
                        metadata[name] = eval(name)
                except NameError:
            #         print(f"Variable {name} not found!") # no need to print not found variables
                    continue
    
            # syn_reversals(cell_type, specs, spine_per_length, frequency, d_lambda, dend_glut, glut_reversal, glutamate_locations, num_gluts, dend_gaba, gaba_reversal, gaba_locations, num_gabas, sim_time, dt=0.025, v_init=-84)

            # Run sim
            i_recording_site, v_recording_site, v_all_3D, Ca_all_3D, imp_all_3D, i_mechs_3D, vspine, v_spine_activated, vdend, \
            v_dend_activated, vsoma, v_dend_tree, v_spine_tree, Ca_spine, Ca_dend, Ca_soma, \
            i_mechs_dend, i_mechs_dend_tree, i_mechs_spine_tree, v_branch, zdend, ztransfer, \
            ampa_currents, nmda_currents, gaba_currents, gaba_conductances, record_dist, \
            record_spine, spine_dist, cap, dt, burn_time, start_time = \
            msNEURONsim(sim_time = sim_time, 
                     stim_time = stim_time,
                     baseline = baseline,
                     glut = glut, 
                     glut_frequency = glut_frequency, 
                     glutamate_locations = glutamate_locations, 
                     glut_reversals = glut_reversals,
                     glut_time = glut_time, 
                     num_gluts = num_gluts, 
                     dend_glut = dend_glut, 
                     g_AMPA = g_AMPA,
                     ratio = ratio,
                     gaba = gaba, 
                     gaba_frequency = gaba_frequency, 
                     gaba_locations = gaba_locations, 
                     gaba_reversals = gaba_reversals,
                     gaba_time = gaba_time, 
                     g_GABA = g_GABA, 
                     num_gabas = num_gabas, 
                     dend_gaba = dend_gaba, 
                     specs = specs, 
                     scale_factors = scale_factors, 
                     gaba_tau1 = gaba_tau1,
                     gaba_tau2 = gaba_tau2,
                     rel_gaba_onsets = rel_gaba_onsets, 
                     rel_glut_onsets = rel_glut_onsets,
                     frequency = frequency,
                     d_lambda = d_lambda,
                     dend2remove = dend2remove,
                     v_init = v_init,
                     AMPA = AMPA,
                     NMDA = NMDA,
                     physiological = physiological,
                     timing_range = timing_range,
                     add_noise = add_noise,
                     beta = beta,                   
                     B = B,                      
                     add_sine = add_sine, 
                     axoshaft = axoshaft,
                     cell_type = cell_type,
                     current_step = current_step,
                     step_current = step_current,
                     step_duration = step_duration,
                     step_start = step_start,
                     holding_current = holding_current,
                     add_ramp = add_ramp,
                     ramp_amplitude = ramp_amplitude,
                     Cm = Cm,
                     Ra = Ra,
                     spine_per_length = spine_per_length,
                     spine_neck_diam = spine_neck_diam,
                     spine_neck_len = spine_neck_len,
                     spine_head_diam = spine_head_diam,
                     soma_diameter = soma_diameter,
                     neck_dynamics = neck_dynamics,
                     space_clamp = space_clamp,
                     record_dendrite = record_dendrite, 
                     record_location = record_location, 
                     record_currents = record_currents,
                     record_branch = record_branch,
                     dend_glut2 = dend_glut2, 
                     record_mechs = record_mechs,
                     mechs2record = mechs2record,
                     record_path_dend = record_path_dend,   
                     record_path_spines = record_path_spines,  
                     record_all_path = record_all_path,
                     record_3D = record_3D,         
                     record_3D_impedance = record_3D_impedance,
                     freq = freq,                   
                     record_3D_mechs = record_3D_mechs,    
                     record_Ca = record_Ca,
                     record_3D_Ca = record_3D_Ca,
                     tonic = tonic,               
                     gbar_gaba = gbar_gaba,
                     rectification = rectification,       
                     distributed = distributed,         
                     gaba_params = gaba_params,
                     tonic_gaba_reversal = tonic_gaba_reversal,
                     dt = dt,
                     ds_imp = ds_imp,
                     voltage_clamp = voltage_clamp,
                     holding_potential = holding_potential,
                     voltage_clamp_site = voltage_clamp_site,
                     voltage_clamp_spine = voltage_clamp_spine,
                     voltage_clamp_loc = voltage_clamp_loc,
                     Rs = Rs,
                     downsample=downsample,
                     ds=ds
            )
    
    
            ind1 = 1 
            ind2 = int(sim_time/deltat)
            t2 = np.arange(1, int(sim_time/deltat), 1) * deltat - deltat 
            soma_v_master.append(go.Scatter(x=t2,  y=extract2(vsoma)[ind1:ind2]))
            dend_v_master.append(go.Scatter(x=t2, y=extract2(vdend)[ind1:ind2]))

            # only do if want to view each sim or save sim graphs
            if record_path_dend:
                plots_return(v_tree=v_dend_tree['v'], vspine=vspine, dists=v_dend_tree['dists'], spine_dist=spine_dist, num_gluts=num_gluts, start_time=start_time, burn_time=burn_time, dt=dt, xaxis_range=[0,500], Nsim_plot=Nsim_plot, Nsim_save=Nsim_save, sim_image_path=sim_image_path, time=time)

            update_data_dictionary(data_dict=data_dict, protocol=protocol, v_all_3D=v_all_3D, 
                    Ca_all_3D=Ca_all_3D, imp_all_3D=imp_all_3D, i_mechs_3D=i_mechs_3D, 
                    vspine=vspine, v_spine_activated=v_spine_activated, vdend=vdend, v_dend_activated=v_dend_activated, 
                    vsoma=vsoma, v_dend_tree=v_dend_tree, v_spine_tree=v_spine_tree, Ca_spine=Ca_spine, 
                    Ca_dend=Ca_dend, Ca_soma=Ca_soma, timing=timing, i_mechs_dend=i_mechs_dend, 
                    i_mechs_dend_tree=i_mechs_dend_tree, i_mechs_spine_tree=i_mechs_spine_tree, v_branch=v_branch, 
                    ampa_currents=ampa_currents, nmda_currents=nmda_currents, gaba_currents=gaba_currents, 
                    gaba_conductances=gaba_conductances, record_dist=record_dist, time=time, 
                    record_currents=record_currents, record_spine=record_spine, spine_dist=spine_dist, cap=cap)
    
            data_dict['metadata'] = metadata
            data_dict['mechanisms_3D_distribution'] = mech_distr_3D
    
            # Save
            if save:
                simulations_path = 'downsample/{}/model{}/{}/simulations/sim{}/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim, ii+1)
                os.makedirs(simulations_path, exist_ok=True)
                names = list(data_dict.keys())
                for name in names:
                    with open('{}/{}.pickle'.format(simulations_path, name), 'wb') as handle:
                        pickle.dump(data_dict[name], handle, protocol=pickle.HIGHEST_PROTOCOL)  

            # # generate plots
            # time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            # sims1 = ['2250', '2254'] 
            # if check_sim(sim, sims1):
            #     plt1(data_dict)

In [ ]:
fig_soma_master, fig_dend_master = plot3(somaV=soma_v_master, dendV=dend_v_master, yaxis='V (mV)', stim_time=120,
        sim_time=sim_time, black_trace=0, gray_trace=1, x_err_bar=50, baseline=20, 
        ds=1, dt=deltat)    

fig_soma_master.show(config={"toImageButtonOptions": {"format": "svg"}})
fig_dend_master.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    sim_image_path = 'downsample/{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim)
    os.makedirs(sim_image_path, exist_ok=True)
    fig_soma_master.write_image(f"{sim_image_path}/fig_soma_master.svg", format='svg', scale=3)
    fig_dend_master.write_image(f"{sim_image_path}/fig_dend_master.svg", format='svg', scale=3)
       